In [79]:
#3. Test that XPU is available

import torch
print(torch.__version__)
print(torch.xpu.is_available())

2.10.0+xpu
True


In [80]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [81]:
#Load Paths

from pathlib import Path

# project root folder
PROJECT_DIR = Path(r"C:\Users\Home\Desktop\projet-Ai")

# dataset folders
SPLIT_DIR = PROJECT_DIR / "data" / "Split Dataset"
IMAGE_DIR = PROJECT_DIR / "data" / "Labelled Images"

# CSV files
TRAIN_CSV = SPLIT_DIR / "Training_meme_dataset.csv"
VAL_CSV = SPLIT_DIR / "validation_meme_dataset.csv"
TEST_CSV = SPLIT_DIR / "Testing_meme_dataset.csv"

# model save location
MODEL_PATH = PROJECT_DIR / "models" / "meme-classification-best_model_xpu.pth"

print("TRAIN CSV:", TRAIN_CSV)
print("VAL CSV:", VAL_CSV)
print("TEST CSV:", TEST_CSV)

print("TRAIN exists:", TRAIN_CSV.exists())
print("VAL exists:", VAL_CSV.exists())
print("TEST exists:", TEST_CSV.exists())

TRAIN CSV: C:\Users\Home\Desktop\projet-Ai\data\Split Dataset\Training_meme_dataset.csv
VAL CSV: C:\Users\Home\Desktop\projet-Ai\data\Split Dataset\validation_meme_dataset.csv
TEST CSV: C:\Users\Home\Desktop\projet-Ai\data\Split Dataset\Testing_meme_dataset.csv
TRAIN exists: True
VAL exists: True
TEST exists: True


In [82]:
# =========================================================
# SETTINGS
# =========================================================
IMAGE_COL = "image_name"
LABEL_COL = "label"

BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 5e-5
NUM_WORKERS = 0

In [83]:
if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: xpu


In [84]:
# =========================================================
# LABEL CLEANING
# =========================================================
def clean_label(x):
    x = str(x).strip().lower()

    if x in ["non-offensiv", "non-offensive", "non offensive", "nonoffensive"]:
        return "non_offensive"
    if x in ["offensive", "offensiv"]:
        return "offensive"

    raise ValueError(f"Unexpected label found: {x}")


label_to_id = {
    "non_offensive": 0,
    "offensive": 1
}

id_to_label = {
    0: "non_offensive",
    1: "offensive"
}

In [85]:
# =========================================================
# LOAD CSV FILES
# =========================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

required_columns = [IMAGE_COL, LABEL_COL]
for col in required_columns:
    if col not in train_df.columns:
        raise ValueError(f"Missing column in training CSV: {col}")
    if col not in val_df.columns:
        raise ValueError(f"Missing column in validation CSV: {col}")
    if col not in test_df.columns:
        raise ValueError(f"Missing column in test CSV: {col}")

Train shape: (445, 3)
Val shape: (149, 3)
Test shape: (149, 3)


In [86]:
# =========================================================
# CLEAN LABELS
# =========================================================
for df in [train_df, val_df, test_df]:
    df[LABEL_COL] = df[LABEL_COL].apply(clean_label)
    df[LABEL_COL] = df[LABEL_COL].map(label_to_id)

print("\nEncoded label counts:")
print("Train:")
print(train_df[LABEL_COL].value_counts().sort_index())
print("Val:")
print(val_df[LABEL_COL].value_counts().sort_index())
print("Test:")
print(test_df[LABEL_COL].value_counts().sort_index())



Encoded label counts:
Train:
label
0    258
1    187
Name: count, dtype: int64
Val:
label
0    91
1    58
Name: count, dtype: int64
Test:
label
0    91
1    58
Name: count, dtype: int64


In [87]:
# =========================================================
# CHECK IMAGE FILES
# =========================================================
def image_exists(image_name):
    image_path = IMAGE_DIR / str(image_name)
    return image_path.exists()

# verify images and optionally remove missing entries
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = df[~df[IMAGE_COL].apply(image_exists)]
    print(f"\nMissing images in {name}: {len(missing)}")
    if len(missing) > 0:
        # show a sample of the missing filenames
        print(missing[[IMAGE_COL]].head(10))
        # drop rows referencing missing files so downstream logic can continue
        df.drop(missing.index, inplace=True)
        print(f"Dropped {len(missing)} rows from {name} split")
        # you can also log or save `missing` if you wish to investigate further



Missing images in train: 0

Missing images in val: 1
    image_name
8  KbTk7Rq.png
Dropped 1 rows from val split

Missing images in test: 0


In [88]:
# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [89]:
# =========================================================
# DATASET
# =========================================================
class MemeDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = str(self.df.loc[idx, IMAGE_COL])
        label = int(self.df.loc[idx, LABEL_COL])

        image_path = self.image_dir / image_name
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [90]:
# =========================================================
# DATALOADERS
# =========================================================
train_dataset = MemeDataset(train_df, IMAGE_DIR, train_transform)
val_dataset = MemeDataset(val_df, IMAGE_DIR, eval_transform)
test_dataset = MemeDataset(test_df, IMAGE_DIR, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [91]:
# =========================================================
# MODEL
# =========================================================
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

class_counts = train_df[LABEL_COL].value_counts().sort_index().values

class_weights = torch.tensor(
    [1.0 / class_counts[0], 1.0 / class_counts[1]],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

use_amp = (device.type == "xpu")

In [92]:
# =========================================================
# TRAINING FUNCTION
# =========================================================
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc



In [93]:
# =========================================================
# EVALUATION FUNCTION
# =========================================================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc, all_labels, all_preds

In [94]:
# =========================================================
# TRAIN LOOP
# =========================================================
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label_to_id": label_to_id,
                "id_to_label": id_to_label
            },
            MODEL_PATH
        )
        print("Best model saved.")

print("\nTraining finished.")
print("Best validation accuracy:", best_val_acc)




Epoch 1/5
Train Loss: 0.7806 | Train Acc: 0.4539
Val   Loss: 0.7435 | Val   Acc: 0.4865
Best model saved.

Epoch 2/5
Train Loss: 0.6061 | Train Acc: 0.6854
Val   Loss: 0.7266 | Val   Acc: 0.5405
Best model saved.

Epoch 3/5
Train Loss: 0.5261 | Train Acc: 0.7573
Val   Loss: 0.7327 | Val   Acc: 0.5676
Best model saved.

Epoch 4/5
Train Loss: 0.4643 | Train Acc: 0.8045
Val   Loss: 0.7605 | Val   Acc: 0.4797

Epoch 5/5
Train Loss: 0.3910 | Train Acc: 0.8584
Val   Loss: 0.7641 | Val   Acc: 0.5405

Training finished.
Best validation accuracy: 0.5675675675675675


In [95]:
# =========================================================
# TEST EVALUATION
# =========================================================
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader)

print("\nTest results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=["non_offensive", "offensive"],
    digits=4
))


Test results
Test Loss: 0.7330
Test Acc : 0.5235

Confusion Matrix:
[[56 35]
 [36 22]]

Classification Report:
               precision    recall  f1-score   support

non_offensive     0.6087    0.6154    0.6120        91
    offensive     0.3860    0.3793    0.3826        58

     accuracy                         0.5235       149
    macro avg     0.4973    0.4973    0.4973       149
 weighted avg     0.5220    0.5235    0.5227       149



In [96]:
#Test the model on a single image

import torch
from torchvision import transforms, models
from PIL import Image
import torch.nn as nn

MODEL_PATH = r"C:\Users\Home\Desktop\projet-Ai\models\meme-classification-best_model_xpu.pth"

IMAGE_PATH = r"C:\Users\Home\Desktop\projet-Ai\data\Labelled Images\6eRZSgn.png"

label_map = {
    0: "non_offensive",
    1: "offensive"
}

device = torch.device("xpu" if torch.xpu.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features,2)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

image = Image.open(IMAGE_PATH).convert("RGB")
image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(image)
    pred = output.argmax(dim=1).item()

print("Prediction:", label_map[pred])

Prediction: non_offensive
